# 04 · Fleet rates, customers vs. their fleets, and persistence

This notebook replicates the Rates by model and age, Customers vs. their fleets, and Persistence and outliers paragraphs of the report's Results section.

> Rates by model and age: The fleet runs at 6.2 unplanned tickets per 100 installed machine-months. The three busiest models (LVD-Remote-2.0, LVRR-WP-2.0, and MV-2.0) hold 42% of machine-months and 51% of tickets, partly because they are younger. scrubBank 480 runs 2.6 times the fleet age for age. Machines under 4 years run about 8 per 100 vs. 2.5 at 10 years or more, but the 10+ rate overstates the drop with age: 83 of its 164 machines with a ticket are Not Serviceable, and removed machines are missing from the list.
>
> Customers vs. their fleets: 63 customers exceed their fleets beyond chance (1,732 tickets vs. 747 expected), and 1 lies outside the normal spread. The top 10% of customers hold 47% of tickets vs. 39% from their fleets plus chance.
>
> Persistence and outliers: Customers above their fleets in the first half tend to stay above them in the second (rank correlation 0.46 vs. 0.04 when shuffled), and machines with 3 or more first-half tickets had 2.6 times their expected tickets in the second half. 18 customers and 36 machines are consistent outliers. Of the 106 chronic outliers on IPA's list, 93 stay flagged, and no flagged machine stands out as a separate group.

**How it works.** `fleet_exposure.py` joins the unplanned tickets to IPA's installed-base list, counts each machine's installed months, fits each machine's expected tickets from its model and age band (over the full period and within each half), and flags customers and machines. We recompute every count, share, rate, and ratio from its per-machine and per-customer tables (`fleet/fx_machine_flags.csv` and `fleet/fx_customer_rates.csv`), and we take installed months by model and by age band from its per-model and per-age-band tables, because the per-machine table lists only the machines with a ticket. The quantities that need the script's fits or resampling over every fleet machine, including those without a ticket, come from its tables: expected tickets (per machine from the main fit and from its variant with service status, per customer, and at the fleet's age-band rates by model), the method-variant flag counts, the negative-binomial q-values, and the chance benchmark for the top 10% of customers, which we redraw with our own seed as a cross-check. The shuffles behind the rank correlation's benchmark are the first draws from the script's random generator, so we replay them with its seed, and we also draw them with 200 other seeds to show how much the benchmark moves. For the reading that the 10+ rate overstates the drop with age, we compare the age-band factors of the script's main fit with those of its variant with service status. The outputs cannot test the reading's second reason, the removed machines. Every cross-check with the script's tables is an assertion, so the notebook stops if one fails.

**Re-run.** With `RERUN = True`, we re-run `verification/fleet_exposure.py`, which takes about 35 s. It rewrites its 20 `fleet/fx_*.csv` tables and `fleet/fx_summary.json` in a scratch copy of `verification/outputs/`, and it writes its real-name table to a scratch copy of `data/private/`, which the notebook does not read and which is deleted when the kernel exits. We then compare each of the 21 files with the committed one.

**Used as committed, without re-running.** The script also reads `htm_device_exposure.csv`, which `high_ticket_machines.py` writes from the telemetry-database extracts (notebook 02 re-runs it). The script takes two things from it: docs/05's 113 chronic outliers and the telemetry-span exposure for its interim telemetry-type rates, which these paragraphs do not use. This notebook reads it to identify the 113. When the optional extract `fleet/db/fxdb_devices.csv` is present, the script bases its telemetry-type rates on installed exposure instead. `fleet_db_extract.py` writes that extract from the telemetry database into `verification/outputs/fleet/db/`, and the committed outputs do not include it. Before the re-run, we delete any local copy from the scratch folder, so the re-run keeps the telemetry-span basis of the committed outputs. `docs/06_fleet_exposure.md` describes the analysis.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    # fleet_exposure.py reads fleet/db/fxdb_devices.csv in the outputs folder it is given (here the scratch copy) and,
    # when that file exists, bases its telemetry-type rates on installed exposure. The committed outputs do not include
    # it, so we delete any local copy from the scratch folder before the re-run.
    assert OUT != R.COMMITTED
    (OUT / 'fleet' / 'db' / 'fxdb_devices.csv').unlink(missing_ok=True)
    R.rerun(OUT, 'fleet_exposure.py')
R.compare(OUT)

fleet_exposure.py: exit status 0, 37 s
    27/27 checks pass
21 files written: 21 identical.


,file written by the re-run,compared with the committed file
0,fleet/fx_age_rates.csv,identical
1,fleet/fx_country_rates.csv,identical
2,fleet/fx_coverage_rates.csv,identical
3,fleet/fx_customer_lorenz.csv,identical
4,fleet/fx_customer_rates.csv,identical
5,fleet/fx_family_rates.csv,identical
6,fleet/fx_machine_flags.csv,identical
7,fleet/fx_match.csv,identical
8,fleet/fx_model_rates.csv,identical
9,fleet/fx_outlier_customers.csv,identical


## The fleet rate: 6.2 per 100 installed machine-months

The fleet rate is the fleet's unplanned tickets divided by its installed machine-months, times 100. `fleet/fx_customer_rates.csv` has one row per customer with a listed machine outside the third-party plan. Each row holds the tickets and installed months of all the customer's machines, including those without a ticket, so we sum the table. We assert that the per-machine table `fleet/fx_machine_flags.csv`, which has one row per fleet machine with a ticket, holds the same tickets, and that `fleet/fx_summary.json` records the same totals.

In [3]:
import math

import numpy as np

customers = R.read_csv(OUT, 'fleet/fx_customer_rates.csv')  # one row per listed customer outside the third-party plan
machines = R.read_csv(OUT, 'fleet/fx_machine_flags.csv')  # one row per fleet machine with an unplanned ticket
summary = R.read_json(OUT, 'fleet/fx_summary.json')
assert customers.customer.is_unique and machines.serial_id.is_unique and (machines.tickets > 0).all()

fleet_tickets = int(customers.tickets.sum())
fleet_months = customers.months.sum()
fleet_rate = 100 * fleet_tickets / fleet_months
n_fleet_machines = int(customers.machines.sum())  # machines with installed months in the period
# Cross-checks: the per-machine table holds the same tickets, and the script's summary records the same totals
assert machines.tickets.sum() == fleet_tickets
assert (summary['fleet']['tickets'], summary['fleet']['machines_exposed']) == (fleet_tickets, n_fleet_machines)
assert math.isclose(summary['fleet']['months'], fleet_months)
assert math.isclose(summary['fleet']['rate_per_100'], fleet_rate)
print(f'{fleet_tickets:,} unplanned tickets on {len(machines):,} of the {n_fleet_machines:,} fleet machines, over '
      f'{fleet_months:,.1f} installed machine-months: {fleet_rate:.3f} per 100. The per-customer table has '
      f'{len(customers):,} customers, {int((customers.months > 0).sum()):,} of them with installed months.')
pd.DataFrame([
    ('fleet/fx_customer_rates.csv, summed over customers', R.num(fleet_tickets), R.num(fleet_months, 1),
     R.num(fleet_rate, 3)),
    ('fleet/fx_machine_flags.csv, summed over machines with a ticket', R.num(machines.tickets.sum()), '', ''),
    ('fleet/fx_summary.json', R.num(summary['fleet']['tickets']), R.num(summary['fleet']['months'], 1),
     R.num(summary['fleet']['rate_per_100'], 3)),
], columns=['source', 'unplanned tickets', 'installed machine-months', 'rate per 100'])

6,750 unplanned tickets on 3,457 of the 8,946 fleet machines, over 109,621.3 installed machine-months: 6.158 per 100. The per-customer table has 1,065 customers, 1,056 of them with installed months.


,source,unplanned tickets,installed machine-months,rate per 100
0,"fleet/fx_customer_rates.csv, summed over customers","6,750","109,621.3",6.158
1,"fleet/fx_machine_flags.csv, summed over machines with a ticket","6,750",,
2,fleet/fx_summary.json,"6,750","109,621.3",6.158


## The three busiest models: 42% of machine-months and 51% of tickets, partly because they are younger

We rank models by their unplanned tickets in the per-machine table and take each model's installed months from `fleet/fx_model_rates.csv`, which has one row per model. To see how much of their higher rate is age, we compare their tickets with their machine-months at the fleet rate (raw) and with the tickets their machines would have at the fleet's age-band rates (age for age), which the script fits jointly with the model factors over every fleet machine (`expected_age_only`). We assert that each model's shares, rate, and ratios equal those in the table.

In [4]:
models = R.read_csv(OUT, 'fleet/fx_model_rates.csv').set_index('model')  # one row per model
model_tickets = machines.groupby('model').tickets.sum().reindex(models.index, fill_value=0)
assert (model_tickets == models.tickets).all() and math.isclose(models.months.sum(), fleet_months)
top3 = list(model_tickets.sort_index().sort_values(ascending=False, kind='mergesort').index[:3])


def vs_fleet(rows):
    """Shares of the fleet and ratios to the fleet's rate for a set of models."""
    t, mo, e = model_tickets[rows].sum(), models.months[rows].sum(), models.expected_age_only[rows].sum()
    return {'share of machine-months': mo / fleet_months, 'share of tickets': t / fleet_tickets,
            'rate per 100': 100 * t / mo, 'raw ratio': (t / mo) / (fleet_tickets / fleet_months),
            'age-for-age ratio': t / e}


busiest = pd.DataFrame({**{model: vs_fleet([model]) for model in top3}, 'the three together': vs_fleet(top3),
                        'fleet': vs_fleet(list(models.index))}).T
script_columns = {'share of machine-months': 'share_of_months', 'share of tickets': 'share_of_tickets',
                  'rate per 100': 'rate_per_100', 'raw ratio': 'ratio_to_fleet', 'age-for-age ratio': 'age_std_ratio'}
assert np.allclose(busiest.loc[top3, list(script_columns)].to_numpy(float),
                   models.loc[top3, list(script_columns.values())].to_numpy(float), rtol=1e-9, atol=0)
share_months, share_tickets, raw_ratio, age_ratio = busiest.loc[
    'the three together', ['share of machine-months', 'share of tickets', 'raw ratio', 'age-for-age ratio']]
if age_ratio <= 1:
    youth = 'entirely because they are younger'
elif age_ratio < raw_ratio:
    youth = 'partly because they are younger'
else:
    youth = 'not because they are younger'
print(f'Together they run {raw_ratio:.2f} times the fleet rate raw and {age_ratio:.2f} times age for age. The fleet\'s '
      f'age-band rates alone give their machines {raw_ratio / age_ratio:.2f} times the fleet rate, so they run above '
      f'the fleet {youth}.')
busiest.round(3)

Together they run 1.21 times the fleet rate raw and 1.08 times age for age. The fleet's age-band rates alone give their machines 1.12 times the fleet rate, so they run above the fleet partly because they are younger.


,share of machine-months,share of tickets,rate per 100,raw ratio,age-for-age ratio
LVD-Remote-2.0,0.157,0.205,8.008,1.301,1.160
LVRR-WP-2.0,0.141,0.177,7.743,1.258,1.122
MV-2.0,0.125,0.131,6.436,1.045,0.932
the three together,0.423,0.512,7.455,1.211,1.080
fleet,1.000,1.000,6.158,1.000,1.000


## scrubBank 480: 2.6 times the fleet, age for age

A model's age-for-age ratio is its tickets divided by the tickets its machines would have at the fleet's age-band rates. That denominator comes from the script's joint model and age fit over every fleet machine, so we read it from `fleet/fx_model_rates.csv` (`expected_age_only`), take the tickets from the per-machine table, and assert that the ratio equals the script's (`age_std_ratio`).

In [5]:
sb = 'scrubBank 480'
sb_ratio = model_tickets[sb] / models.expected_age_only[sb]
assert math.isclose(sb_ratio, models.age_std_ratio[sb], rel_tol=1e-9)
print(f"{sb}: {model_tickets[sb]} unplanned tickets vs. {models.expected_age_only[sb]:.1f} at the fleet's age-band "
      f'rates, {sb_ratio:.3f} times the fleet age for age (95% interval in fx_model_rates.csv: '
      f'{models.age_std_lo[sb]:.2f}–{models.age_std_hi[sb]:.2f}; raw: {models.ratio_to_fleet[sb]:.2f} times).')

scrubBank 480: 122 unplanned tickets vs. 47.3 at the fleet's age-band rates, 2.581 times the fleet age for age (95% interval in fx_model_rates.csv: 1.75–3.79; raw: 2.66 times).


## Machines under 4 years: about 8 per 100 vs. 2.5 at 10 years or more

Age is the years from install to the midpoint of a machine's installed time in the period, in the script's bands (<1, 1–2, 2–4, 4–6, 6–8, 8–10, and 10+ years). Under 4 years and 10 years or more are the report's cuts, which we apply to the band labels: under 4 years covers the bands <1, 1–2, and 2–4 (installed after about Feb 15th, 2022, in the report's Definitions), and 10 years or more covers the band 10+. We count each band's tickets in the per-machine table, take its installed months from `fleet/fx_age_rates.csv`, which has one row per band, and assert that each band's tickets and rate equal the table's.

In [6]:
ages = R.read_csv(OUT, 'fleet/fx_age_rates.csv').set_index('age_band')  # one row per age band, youngest first
band_tickets = machines.groupby('age_band').tickets.sum().reindex(ages.index, fill_value=0)
assert (band_tickets == ages.tickets).all() and math.isclose(ages.months.sum(), fleet_months)
band_rate = 100 * band_tickets / ages.months
assert np.allclose(band_rate, ages.rate_per_100, rtol=1e-9, atol=0)
YOUNG, OLD = 4, 10  # years: the report's cuts, under 4 years and 10 years or more


def band_years(label):
    """A band's years from its label: '<1' -> (0, 1), '2-4' -> (2, 4), '10+' -> (10, inf)."""
    if label.startswith('<'):
        return 0.0, float(label[1:])
    if label.endswith('+'):
        return float(label[:-1]), math.inf
    lo, hi = label.split('-')
    return float(lo), float(hi)


years = {band: band_years(band) for band in ages.index}
assert all(hi <= cut or lo >= cut for lo, hi in years.values() for cut in (YOUNG, OLD))  # no band spans a cut
young_bands = [band for band, (lo, hi) in years.items() if hi <= YOUNG]
old_bands = [band for band, (lo, hi) in years.items() if lo >= OLD]
assert young_bands and old_bands
rate_young = 100 * band_tickets[young_bands].sum() / ages.months[young_bands].sum()
rate_old = 100 * band_tickets[old_bands].sum() / ages.months[old_bands].sum()
print(f'Under {YOUNG} years ({", ".join(young_bands)}): {rate_young:.2f} per 100; {OLD} years or more '
      f'({", ".join(old_bands)}): {rate_old:.2f} per 100; fleet: {fleet_rate:.2f} per 100.')
pd.DataFrame({'machines': ages.machines, 'unplanned tickets': band_tickets,
              'installed machine-months': ages.months.round(0).astype(int), 'rate per 100': band_rate.round(2),
              'vs. the fleet rate': np.where(band_rate > fleet_rate, 'above', 'below')})

Under 4 years (<1, 1-2, 2-4): 8.05 per 100; 10 years or more (10+): 2.53 per 100; fleet: 6.16 per 100.


,machines,unplanned tickets,installed machine-months,rate per 100,vs. the fleet rate
age_band,,,,,
<1,1497,1071,12708,8.43,above
1-2,866,882,11267,7.83,above
2-4,1624,1676,21129,7.93,above
4-6,1731,1125,22521,5.00,below
6-8,1458,1038,18969,5.47,below
8-10,939,684,12217,5.60,below
10+,831,274,10812,2.53,below


## The 10+ band: 83 of its 164 machines with a ticket are Not Serviceable

Service status is the list's Oct 19th, 2026, snapshot. We count the 10+ band's machines with a ticket by status in the per-machine table and assert that their total equals the band's `machines_with_ticket` in `fleet/fx_age_rates.csv`. The list holds only machines active on Oct 19th, 2026, so removed machines are missing, and we print the unplanned tickets on serials that are not on the list from `fleet/fx_match.csv` as context.

In [7]:
in_old = machines[machines.age_band.isin(old_bands)]  # the band's machines with a ticket
n_old = len(in_old)
n_old_not_serviceable = int((in_old.service_status == 'Not Serviceable').sum())
assert n_old == ages.machines_with_ticket[old_bands].sum()
match = R.read_csv(OUT, 'fleet/fx_match.csv').set_index('measure')['value']
print(f'{n_old_not_serviceable} of the {n_old} machines in the {", ".join(old_bands)} band with a ticket are Not '
      f'Serviceable.')
print(f"Not on the list: {match['unplanned tickets on unlisted serials']:,} unplanned tickets on "
      f"{match['unlisted serials with unplanned tickets']:,} serials, which drop out with their unknown exposure.")
(in_old.service_status.value_counts().rename(f'{", ".join(old_bands)} machines with a ticket')
 .rename_axis('service status').to_frame())

83 of the 164 machines in the 10+ band with a ticket are Not Serviceable.
Not on the list: 433 unplanned tickets on 226 serials, which drop out with their unknown exposure.


,10+ machines with a ticket
service status,
Not Serviceable,83
No Coverage,44
Service Contract,37


**The reading that the 10+ rate overstates the drop with age.** The report gives the two facts above as its reasons. The script's variant with service status tests the first. Its main fit leaves service status out, because coverage is partly a customer's choice and the status is a snapshot (docs/06), but one of its method variants refits expected tickets over every fleet machine, including those without a ticket, with service status as a third factor beside model group and age band (`expected_with_coverage` in the per-machine table). Each fit gives a machine's expected tickets as its installed months times one factor for each of its levels, so the logarithm of its expected tickets per installed month is a sum of one term per level. Least squares on the machines with a ticket recovers the age-band factors, and we assert that they reproduce every machine's expected tickets. If service status accounts for part of the 10+ band's low rate, the band's factor relative to the bands under 4 years is higher in the variant than in the main fit, and the check has a row for that comparison. The outputs cannot test the second reason: removed machines are not on the list, so the outputs give them no age band.

In [8]:
def band_factors(column, factors):
    """Age-band factors of one of the script's fits, relative to the youngest band, recovered from the machines with a
    ticket: the log of a machine's expected tickets per installed month is a sum of one term per level it has."""
    X = pd.get_dummies(machines[factors].astype(str), dtype=float)
    y = np.log(machines[column] / machines.months).to_numpy()
    beta, _, rank, _ = np.linalg.lstsq(X.to_numpy(), y, rcond=None)
    assert rank == X.shape[1] - (len(factors) - 1)  # only the split of the scale between factors is free
    assert np.abs(X.to_numpy() @ beta - y).max() < 1e-9  # the terms reproduce every machine's expected tickets
    term = pd.Series(beta, X.columns)[[f'age_band_{band}' for band in ages.index]].to_numpy()
    return pd.Series(np.exp(term - term[0]), ages.index)


assert (machines.months > 0).all() and (machines.expected_with_coverage > 0).all()
fits = pd.DataFrame({
    'model and age (the main fit)': band_factors('expected', ['rate_group', 'age_band']),
    'model, age, and service status (the variant)': band_factors('expected_with_coverage',
                                                                 ['rate_group', 'age_band', 'service_status']),
}).T.rename_axis(columns=f'age-band factor relative to {ages.index[0]}')
(oldest,) = old_bands  # 10 years or more is a single band
vs_young = fits[[oldest]].to_numpy() / fits[young_bands].to_numpy()  # the oldest band vs. each band under 4 years
rises = bool((vs_young[1] > vs_young[0] * (1 + 1e-6)).all())  # beyond rounding: status explains part of the drop
reading = 'overstates the drop with age' if rises else 'does not overstate the drop with age'
print(f"Relative to each band under {YOUNG} years, the {oldest} band's factor is "
      f'{R.span(vs_young[0].min(), vs_young[0].max(), 2)} in the main fit and '
      f'{R.span(vs_young[1].min(), vs_young[1].max(), 2)} with service status in the fit, so service status accounts '
      f"for {'part' if rises else 'none'} of the band's low rate, and the {oldest} rate {reading}.")
fits.round(3)

Relative to each band under 4 years, the 10+ band's factor is 0.35–0.37 in the main fit and 0.94–0.95 with service status in the fit, so service status accounts for part of the band's low rate, and the 10+ rate overstates the drop with age.


age-band factor relative to <1,<1,1-2,2-4,4-6,6-8,8-10,10+
model and age (the main fit),1.0,0.930,0.954,0.604,0.656,0.731,0.348
"model, age, and service status (the variant)",1.0,0.985,0.990,0.653,0.709,0.824,0.940


## 63 customers above their fleets beyond chance: 1,732 tickets vs. 747 expected

A customer's expected tickets are the sum of its machines' fitted expected tickets for their model, age band, and installed months (`expected` in `fleet/fx_customer_rates.csv`). We recompute each customer's one-sided exact Poisson p-value from its tickets and expected tickets, apply Benjamini–Hochberg over the customers with expected tickets, and flag those with q < 0.05 and at least 5 tickets. We assert that our q-values and flags equal the table's and that our count equals the one in `fleet/fx_summary.json`.

In [9]:
def poisson_upper(k, mu):
    """P(X >= k) for X ~ Poisson(mu) with mu > 0, summed over the upper tail so tiny p-values stay accurate."""
    k = int(k)
    if k <= 0:
        return 1.0
    if k <= mu:
        return max(0.0, 1.0 - sum(math.exp(-mu + j * math.log(mu) - math.lgamma(j + 1)) for j in range(k)))
    term = math.exp(-mu + k * math.log(mu) - math.lgamma(k + 1))
    total, j = term, k
    while term > total * 1e-17:
        j += 1
        term *= mu / j
        total += term
    return min(1.0, total)


def upper_p(observed, expected):
    """One-sided Poisson p-values; NaN where nothing is expected (those units are not tested)."""
    return np.array([poisson_upper(o, e) if e > 0 else np.nan for o, e in zip(observed, expected)])


def bh(p):
    """Benjamini–Hochberg q-values over the p-values that are not NaN."""
    p = np.asarray(p, float)
    q = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    n = int(ok.sum())
    order = np.argsort(p[ok], kind='mergesort')
    ranked = p[ok][order] * n / np.arange(1, n + 1)
    q_ok = np.empty(n)
    q_ok[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    q[ok] = q_ok
    return q


p_customer = upper_p(customers.tickets, customers.expected)
q_customer = bh(p_customer)
flagged_c = (customers.tickets >= 5) & (q_customer < 0.05)
assert np.allclose(q_customer, customers.q_poisson, equal_nan=True) and (flagged_c == customers.poisson_flagged).all()
n_flagged = int(flagged_c.sum())
assert n_flagged == summary['customers']['poisson_flagged']
flagged_tickets = int(customers.tickets[flagged_c].sum())
flagged_expected = customers.expected[flagged_c].sum()
print(f'{n_flagged} of the {int((customers.expected > 0).sum()):,} customers with expected tickets are flagged: '
      f'{flagged_tickets:,} tickets vs. {flagged_expected:.1f} expected ({flagged_tickets / fleet_tickets:.1%} of all '
      f'tickets).')
(customers.assign(group=np.where(flagged_c, 'flagged beyond chance', 'not flagged'))
 .groupby('group').agg(customers=('customer', 'size'), tickets=('tickets', 'sum'), expected=('expected', 'sum'))
 .assign(STR=lambda d: d.tickets / d.expected).rename_axis(None).round(2))

63 of the 1,056 customers with expected tickets are flagged: 1,732 tickets vs. 747.4 expected (25.7% of all tickets).


,customers,tickets,expected,STR
flagged beyond chance,63,1732,747.42,2.32
not flagged,1002,5018,6002.58,0.84


## 1 customer outside the normal spread

The overdispersion-adjusted funnel scores each customer on the square-root scale, z_D = (√(O/E) − 1) / √(1/(4E) + τ²), where τ² is the spread between customers, a moment estimate from the unadjusted scores winsorized at their 10th and 90th percentiles. A customer lies outside the normal spread when z_D is above the funnel's 99.8% limit (3.090), and because τ² depends only on each customer's tickets and expected tickets in `fleet/fx_customer_rates.csv`, we recompute it. We assert that τ² and the count of customers outside the normal spread equal those in `fleet/fx_summary.json`.

In [10]:
def funnel(observed, expected, winsor=0.10):
    """Overdispersion-adjusted funnel scores z_D and tau^2 (Spiegelhalter 2005, square-root scale)."""
    O, E = np.asarray(observed, float), np.asarray(expected, float)
    ok = E > 0
    y, s = np.sqrt(O[ok] / E[ok]), 1 / (2 * np.sqrt(E[ok]))
    z = (y - 1) / s
    zw = np.clip(z, *np.percentile(z, [100 * winsor, 100 * (1 - winsor)]))
    w, n = 1 / s ** 2, int(ok.sum())
    tau2 = max(0.0, (n * np.mean(zw ** 2) - (n - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
    zd = np.full(len(O), np.nan)
    zd[ok] = (y - 1) / np.sqrt(s ** 2 + tau2)
    return zd, tau2


z_d, tau2 = funnel(customers.tickets, customers.expected)
z95, z998 = summary['definitions']['z95'], summary['definitions']['z998']
beyond = z_d > z998
n_beyond = int(beyond.sum())
assert math.isclose(tau2, summary['customers']['tau2'], rel_tol=1e-9)
assert np.allclose(z_d, customers.z_d, equal_nan=True) and (beyond == customers.beyond_funnel_998).all()
assert n_beyond == summary['customers']['flagged_998']
print(f'tau^2 = {tau2:.5f}, as in fx_summary.json.')
pd.Series({'above the 99.8% limit (outside the normal spread)': n_beyond, 'above the 95% limit': int((z_d > z95).sum()),
           'below the lower 99.8% limit': int((z_d < -z998).sum())}, name='customers').to_frame()

tau^2 = 0.09040, as in fx_summary.json.


,customers
above the 99.8% limit (outside the normal spread),1
above the 95% limit,32
below the lower 99.8% limit,0


## The top 10% of customers hold 47% of tickets vs. 39% from their fleets plus chance

We rank the customers with installed months by unplanned tickets and take the share of tickets held by the top 10% on the script's curve, which interpolates between whole customers, so the observed share and the chance benchmark have the same base. We assert that our observed share and the share their expected tickets alone would give equal those in `fleet/fx_customer_lorenz.csv` at 10% of customers. The chance benchmark is the median over the script's 1,000 draws of every customer's tickets from a Poisson distribution with its expected tickets, so we read it from that table. The script makes those draws after it shuffles first-half tickets among machines, including the machines without a ticket, which the outputs do not list, so we cannot replay them. We redraw them with our own seed and assert that our median and percentiles are within 0.3 points of the script's.

In [11]:
TOP = 0.10  # the report's top 10% of customers


def top_share(x):
    """Share of the total held by the top 10% of units, ranked by value and interpolated between whole units, as on
    the script's curve in fx_customer_lorenz.csv."""
    x = np.sort(np.asarray(x, float))[::-1]
    curve = np.concatenate([[0.0], np.cumsum(x) / x.sum()])
    return float(np.interp(TOP, np.linspace(0, 1, len(x) + 1), curve))


active = customers[customers.months > 0]  # the customers with installed months
n_active = len(active)
top_obs, top_expected = top_share(active.tickets), top_share(active.expected)
lorenz = R.read_csv(OUT, 'fleet/fx_customer_lorenz.csv')  # the script's top shares at each 1% of customers
at10 = lorenz[np.isclose(lorenz.customer_share, TOP)].iloc[0]
chance_share = at10.ticket_share_null_median
draws = np.random.default_rng(2026).poisson(active.expected.to_numpy(), size=(summary['n_null'], n_active))
redraw = np.array([top_share(d) for d in draws])
assert math.isclose(top_obs, at10.ticket_share_obs, rel_tol=1e-9)
assert math.isclose(top_expected, at10.expected_share, rel_tol=1e-9)
# Within 0.3 points: other seeds move the median by about 0.02 points and the percentiles by about 0.05
chance_columns = ['ticket_share_null_median', 'ticket_share_null_p2_5', 'ticket_share_null_p97_5']
assert np.allclose([np.median(redraw), *np.percentile(redraw, [2.5, 97.5])], at10[chance_columns].astype(float),
                   rtol=0, atol=0.003)
k = math.ceil(TOP * n_active)
whole = np.sort(active.tickets.to_numpy())[::-1][:k].sum() / active.tickets.sum()
print(f'The top 10% of the {n_active:,} customers with installed months ({TOP * n_active:.1f} customers) hold '
      f'{top_obs:.1%} of their unplanned tickets on the curve; the top {k} whole customers hold {whole:.1%}.')
pd.DataFrame({
    'observed': [at10.ticket_share_obs, top_obs],
    'expected tickets (fleets alone)': [at10.expected_share, top_expected],
    'fleets plus chance: median': [chance_share, np.median(redraw)],
    'fleets plus chance: 2.5th percentile': [at10.ticket_share_null_p2_5, np.percentile(redraw, 2.5)],
    'fleets plus chance: 97.5th percentile': [at10.ticket_share_null_p97_5, np.percentile(redraw, 97.5)],
}, index=['fx_customer_lorenz.csv (the script)', 'recomputed here (our draws)']).map(lambda x: R.pct(x, 1))

The top 10% of the 1,056 customers with installed months (105.6 customers) hold 47.1% of their unplanned tickets on the curve; the top 106 whole customers hold 47.2%.


,observed,expected tickets (fleets alone),fleets plus chance: median,fleets plus chance: 2.5th percentile,fleets plus chance: 97.5th percentile
fx_customer_lorenz.csv (the script),47.1%,37.9%,39.2%,38.1%,40.3%
recomputed here (our draws),47.1%,37.9%,39.2%,38.2%,40.2%


## Customers stay above their fleets: rank correlation 0.46 vs. 0.04 when shuffled

Each half's STR is a customer's tickets in that half divided by its expected tickets refitted within the half (`O_h1`, `E_h1`, `O_h2`, and `E_h2` in `fleet/fx_customer_rates.csv`), and the rank correlation covers the customers with at least 3 expected tickets in each half. The shuffled value is the median over the script's 1,000 permutations that shuffle second-half STRs among customers in the same tenth of second-half expected tickets. These permutations are the first draws from the script's random generator, so we replay them with its seed and take the median of our replay. We assert that our rank correlation, and our replay's median and percentiles, equal those in `fleet/fx_outlier_persistence.csv`.

In [12]:
persistence = R.read_csv(OUT, 'fleet/fx_outlier_persistence.csv').set_index('statistic')
both_halves = customers[(customers.E_h1 >= 3) & (customers.E_h2 >= 3)].sort_values('customer')  # the script's order
str_h1 = (both_halves.O_h1 / both_halves.E_h1).to_numpy()
str_h2 = (both_halves.O_h2 / both_halves.E_h2).to_numpy()
# Spearman rank correlation: the correlation of the ranks, with tied values sharing their average rank. A shuffle
# moves each second-half rank with its STR, so we rank once and correlate the centered ranks.
rank_h1, rank_h2 = (pd.Series(s).rank().to_numpy() for s in (str_h1, str_h2))
dev_h1, dev_h2 = rank_h1 - rank_h1.mean(), rank_h2 - rank_h2.mean()
scale = np.sqrt((dev_h1 @ dev_h1) * (dev_h2 @ dev_h2))
rho = float(dev_h1 @ dev_h2 / scale)
tenth = pd.qcut(both_halves.E_h2.rank(method='first'), 10, labels=False).to_numpy()  # tenths of E_h2, as the script
tenths = [np.where(tenth == g)[0] for g in range(10)]


def shuffled(seed):
    """Rank correlations of 1,000 shuffles drawn as the script draws them, from a generator with this seed: each
    shuffle permutes the second-half STRs among the customers in each tenth of second-half expected tickets."""
    rng = np.random.default_rng(seed)
    idx = np.tile(np.arange(len(both_halves)), (summary['n_null'], 1))  # one row of customer positions per shuffle
    for row in idx:
        for sel in tenths:
            row[sel] = rng.permutation(sel)
    return dev_h2[idx] @ dev_h1 / scale


replay = shuffled(summary['seed'])  # the script's own draws
rho_null = float(np.median(replay))
spearman_row = persistence.loc['spearman_STR_h1_h2']
columns = ['observed', 'perm_null_median', 'perm_null_p2_5', 'perm_null_p97_5']
recomputed = [rho, rho_null, *np.percentile(replay, [2.5, 97.5])]
assert spearman_row.units == len(both_halves)
assert np.allclose(recomputed, spearman_row[columns].astype(float), rtol=0, atol=1e-9)
print(f'{len(both_halves)} customers with at least 3 expected tickets in each half: rank correlation {rho:.4f}; '
      f"the median of the script's 1,000 shuffles, replayed with its seed: {rho_null:.4f}.")
pd.DataFrame([spearman_row[columns].astype(float).to_numpy(), recomputed],
             columns=['observed', 'shuffled: median', 'shuffled: 2.5th percentile', 'shuffled: 97.5th percentile'],
             index=['fx_outlier_persistence.csv', "recomputed here (replay with the script's seed)"]).round(4)

311 customers with at least 3 expected tickets in each half: rank correlation 0.4589; the median of the script's 1,000 shuffles, replayed with its seed: 0.0366.


,observed,shuffled: median,shuffled: 2.5th percentile,shuffled: 97.5th percentile
fx_outlier_persistence.csv,0.4589,0.0366,-0.071,0.1468
recomputed here (replay with the script's seed),0.4589,0.0366,-0.071,0.1468


**The second decimal of the shuffled value depends on the seed.** A median over 1,000 shuffles carries Monte Carlo noise, so another seed gives a slightly different value. We draw the 1,000 shuffles again with each of 200 other seeds (1 to 200), and we take each run's median and the median over all their shuffles together. The check keeps its row for the shuffled value, which tests the script's own draws, and the table below lists the value with its spread over the other seeds.

In [13]:
other_runs = np.array([shuffled(seed) for seed in range(1, 201)])  # 200 other seeds, 1,000 shuffles each
other = np.median(other_runs, axis=1)
pooled = float(np.median(other_runs))
printed = pd.Series([R.num(x, 2) for x in other]).value_counts().sort_index()  # each run's median, as printed
counts = [f'{n} print {value}' for value, n in printed.items()]
runs = ' and '.join(counts) if len(counts) <= 2 else ', '.join(counts[:-1]) + ', and ' + counts[-1]
print(f'With 200 other seeds, the medians run from {other.min():.4f} to {other.max():.4f} (standard deviation '
      f'{other.std(ddof=1):.4f}): {runs}. Over all {other_runs.size:,} of their shuffles, the median is {pooled:.4f}.')
report_null = '0.04'  # the report's shuffled value
differ = sorted(set(printed.index) - {report_null})  # what the other seeds print instead
n_differ = int(printed.drop(report_null, errors='ignore').sum())
pd.DataFrame([{'Finding': 'Rank correlation when shuffled (median of 1,000 permutations)', 'Report': report_null,
               'What the outputs give': f"{rho_null:.4f} with the script's seed; {R.span(other.min(), other.max(), 4)} "
                                        f'with 200 other seeds ({runs}); {pooled:.4f} over all their shuffles',
               'Reproduced': f"{'yes' if R.num(rho_null, 2) == report_null else 'no'} with the script's seed and "
                             f"{'yes' if R.num(pooled, 2) == report_null else 'no'} over all the other seeds' "
                             f'shuffles; ' + (f"{n_differ} of the 200 other seeds print {' or '.join(differ)}" if differ
                                              else 'every other seed prints the same value')}])

With 200 other seeds, the medians run from 0.0299 to 0.0425 (standard deviation 0.0021): 46 print 0.03 and 154 print 0.04. Over all 200,000 of their shuffles, the median is 0.0365.


,Finding,Report,What the outputs give,Reproduced
0,"Rank correlation when shuffled (median of 1,000 permutations)",0.04,0.0366 with the script's seed; 0.0299–0.0425 with 200 other seeds (46 print 0.03 and 154 print 0.04); 0.0365 over all their shuffles,yes with the script's seed and yes over all the other seeds' shuffles; 46 of the 200 other seeds print 0.03


## Machines with 3 or more first-half tickets: 2.6 times their expected tickets in the second half

For the machines with installed time in both halves, we take those with at least 3 first-half tickets, the report's cut, and divide their second-half tickets by their second-half expected tickets, refitted within the half, from the per-machine table. We assert that the ratio equals the script's statistic for that cut in `fleet/fx_outlier_persistence.csv`, and that our ratios by first-half tickets equal the rows of `fleet/fx_outlier_machine_persistence.csv`.

In [14]:
CUT = 3  # first-half tickets: the report's cut, 3 or more
stat = f'second_half_ratio_if_{CUT}plus_first_half'  # the script's statistic for that cut
in_both = machines[(machines.months_h1 > 0) & (machines.months_h2 > 0)]
picked = in_both[in_both.tickets_h1 >= CUT]
machine_ratio = picked.tickets_h2.sum() / picked.expected_h2.sum()
assert len(picked) == persistence.units[stat]
assert math.isclose(machine_ratio, persistence.observed[stat], rel_tol=1e-9)
print(f'{len(picked)} machines with {CUT} or more first-half tickets: {picked.tickets_h2.sum()} second-half tickets '
      f'vs. {picked.expected_h2.sum():.1f} expected, {machine_ratio:.4f} times (first-half counts shuffled within '
      f'model and age cells give {persistence.perm_null_median[stat]:.2f}).')

groups = R.read_csv(OUT, 'fleet/fx_outlier_machine_persistence.csv', dtype={'first_half_tickets': str})
groups = groups.set_index('first_half_tickets').second_half_ratio
rows = {'1': in_both.tickets_h1 == 1, '2': in_both.tickets_h1 == 2, '3': in_both.tickets_h1 == 3,
        '4+': in_both.tickets_h1 >= 4, f'{CUT} or more': in_both.tickets_h1 >= CUT}
by_count = pd.DataFrame([{'machines': int(sel.sum()), 'second-half tickets': int(in_both.tickets_h2[sel].sum()),
                          'second-half expected': in_both.expected_h2[sel].sum(),
                          'ratio': in_both.tickets_h2[sel].sum() / in_both.expected_h2[sel].sum(),
                          "ratio in the script's tables": groups.get(name, persistence.observed[stat])}
                         for name, sel in rows.items()], index=pd.Index(list(rows), name='first-half tickets'))
assert np.allclose(by_count.ratio, by_count["ratio in the script's tables"], rtol=1e-9, atol=0)
by_count.round(3)

272 machines with 3 or more first-half tickets: 334 second-half tickets vs. 128.8 expected, 2.5932 times (first-half counts shuffled within model and age cells give 1.05).


,machines,second-half tickets,second-half expected,ratio,ratio in the script's tables
first-half tickets,,,,,
1,1407,827,613.098,1.349,1.349
2,421,302,187.366,1.612,1.612
3,142,189,66.617,2.837,2.837
4+,130,145,62.180,2.332,2.332
3 or more,272,334,128.797,2.593,2.593


## 18 customers and 36 machines are consistent outliers

A consistent outlier is flagged in at least 10 of the 12 method variants and has a one-sided Poisson p < 0.05 in each half. Each variant refits expected tickets for every fleet machine, including those without a ticket, so we read how many variants flag each customer and machine (`variants_flagged`) from the per-customer and per-machine tables, and we recompute the half p-values from each half's tickets and expected tickets. We assert that our consistent outliers are those the tables mark and that our counts equal those in `fleet/fx_summary.json`.

In [15]:
min_variants, n_variants = summary['definitions']['min_variants'], summary['definitions']['n_variants']
p1_c, p2_c = upper_p(customers.O_h1, customers.E_h1), upper_p(customers.O_h2, customers.E_h2)
consistent_c = (customers.variants_flagged >= min_variants) & (p1_c < 0.05) & (p2_c < 0.05)
p1_m, p2_m = upper_p(machines.tickets_h1, machines.expected_h1), upper_p(machines.tickets_h2, machines.expected_h2)
consistent_m = (machines.variants_flagged >= min_variants) & (p1_m < 0.05) & (p2_m < 0.05)
assert (consistent_c == customers.consistent_outlier).all() and (consistent_m == machines.consistent_outlier).all()
n_consistent_c, n_consistent_m = int(consistent_c.sum()), int(consistent_m.sum())
assert (n_consistent_c, n_consistent_m) == (summary['customers']['consistent_outliers'],
                                            summary['machines']['consistent_outliers'])
pd.DataFrame([{f'flagged in {min_variants}+ of {n_variants} variants': int((t.variants_flagged >= min_variants).sum()),
               'consistent outliers': int(c.sum()), 'their tickets': int(t.tickets[c].sum()),
               'their expected tickets': round(float(t.expected[c].sum()), 1)}
              for t, c in ((customers, consistent_c), (machines, consistent_m))], index=['customers', 'machines'])

,flagged in 10+ of 12 variants,consistent outliers,their tickets,their expected tickets
customers,32,18,737,318.4
machines,58,36,310,31.3


## Of the 106 chronic outliers on IPA's list, 93 stay flagged

The chronic outliers of docs/05 are the 113 devices labeled `chronic_outlier` in `htm_device_exposure.csv` (notebook 02 rebuilds them). One is on IPA's list when its serial is in the per-machine table, which covers every listed machine that has a ticket or is among the 113, apart from the third-party plan, which holds none of them, as we assert from `fleet/fx_summary.json`. A machine stays flagged when it has at least 5 unplanned tickets and a Benjamini–Hochberg q < 0.05 over every fleet machine with installed months, which we recompute from its tickets and expected tickets, counting each machine without a ticket at p = 1. We assert that our flags equal the table's and that our counts equal those in `fleet/fx_summary.json`.

In [16]:
htm = R.read_csv(OUT, 'htm_device_exposure.csv', usecols=['serial_id', 'class'])  # one row per telemetry device
chronic = set(htm.serial_id[htm['class'] == 'chronic_outlier'])
overlap = summary['docs05_overlap']
# The per-machine table holds every chronic outlier on IPA's list only if none is on the third-party plan
assert len(chronic) == overlap['docs05_flagged'] and overlap['on_list_third_party'] == 0
# Every rate group and age band has tickets, so every fleet machine with installed months has expected tickets above 0
assert (models.tickets[models.own_rate] > 0).all() and models.tickets[~models.own_rate].sum() > 0
assert (ages.tickets > 0).all() and (machines.expected > 0).all()
p_machine = upper_p(machines.tickets, machines.expected)
q_machine = bh(np.concatenate([p_machine, np.ones(n_fleet_machines - len(machines))]))[:len(machines)]
flagged_m = (machines.tickets >= 5) & (q_machine < 0.05)
assert np.allclose(q_machine, machines.q_poisson) and (flagged_m == machines.flagged).all()
on_list = machines.serial_id.isin(chronic)
assert (on_list == machines.in_docs05_113).all()
n_on_list = int(on_list.sum())
n_still_flagged = int((on_list & flagged_m).sum())
assert int(flagged_m.sum()) == summary['machines']['flagged']
assert (n_on_list, n_still_flagged, int((on_list & consistent_m).sum())) == (
    overlap['in_population'], overlap['flagged_now'], overlap['consistent_now'])
print(f'{int(flagged_m.sum())} fleet machines are flagged, and none of the {len(chronic)} chronic outliers is on the '
      f'third-party plan.')
pd.Series({'chronic outliers (docs/05)': len(chronic), "on IPA's list": n_on_list, 'still flagged': n_still_flagged,
           'consistent outliers': int((on_list & consistent_m).sum())}, name='machines').to_frame()

128 fleet machines are flagged, and none of the 113 chronic outliers is on the third-party plan.


,machines
chronic outliers (docs/05),113
on IPA's list,106
still flagged,93
consistent outliers,30


## No flagged machine stands out as a separate group

The negative-binomial check lets each model group's ticket rate vary between machines, with a dispersion that the script fits over every fleet machine, including those without a ticket. We read each machine's negative-binomial q-value (`q_nb`) from the per-machine table and count the flagged machines that still have q < 0.05. We also assert that the table gives the count in `fleet/fx_summary.json`, which covers every machine with at least 5 tickets.

In [17]:
n_nb = int((flagged_m & (machines.q_nb < 0.05)).sum())
n_nb_any = int(((machines.tickets >= 5) & (machines.q_nb < 0.05)).sum())  # fx_summary.json's count
assert n_nb_any == summary['machines']['flagged_nb']
print(f'{n_nb} of the {int(flagged_m.sum())} flagged machines have a negative-binomial q < 0.05; the smallest q among '
      f'them is {machines.q_nb[flagged_m].min():.3f}. Among all machines with at least 5 tickets, {n_nb_any} have '
      f'q < 0.05.')

0 of the 128 flagged machines have a negative-binomial q < 0.05; the smallest q among them is 0.103. Among all machines with at least 5 tickets, 0 have q < 0.05.


## Check against the report

The cuts the report names (under 4 years, 10 years or more, the top 10%, and 3 or more first-half tickets) are definitions we apply, so the table checks the results they give. The row for the reading that the 10+ rate overstates the drop with age tests its first reason, service status, with the script's method variant that adds service status to the fit. The outputs cannot test the second reason, the removed machines. The row for the shuffled rank correlation checks the script's own draws, and the table after the rank correlation shows that the second decimal of that value depends on the seed.

In [18]:
R.check([
    ('Fleet rate: unplanned tickets per 100 installed machine-months', '6.2', R.num(fleet_rate, 1)),
    ('Three busiest models by unplanned tickets', 'LVD-Remote-2.0, LVRR-WP-2.0, and MV-2.0',
     ', '.join(top3[:-1]) + ', and ' + top3[-1]),
    ('Their share of installed machine-months', '42%', R.pct(share_months)),
    ('Their share of unplanned tickets', '51%', R.pct(share_tickets)),
    ('Their excess: raw vs. age-for-age ratio', 'partly because they are younger', youth),
    ('scrubBank 480 vs. the fleet, age for age (times)', '2.6', R.num(sb_ratio, 1)),
    ('Rate per 100: machines under 4 years', '8', R.num(rate_young)),
    ('Rate per 100: machines 10 years or more', '2.5', R.num(rate_old, 1)),
    ('The 10+ rate vs. the drop with age (age-band factors with vs. without service status in the fit)',
     'overstates the drop with age', reading),
    ('10+ machines with a ticket that are Not Serviceable', '83', R.num(n_old_not_serviceable)),
    ('10+ machines with a ticket', '164', R.num(n_old)),
    ('Customers above their fleets beyond chance', '63', R.num(n_flagged)),
    ('Their unplanned tickets', '1,732', R.num(flagged_tickets)),
    ('Their expected tickets', '747', R.num(flagged_expected)),
    ('Customers outside the normal spread (above the 99.8% funnel limit)', '1', R.num(n_beyond)),
    ("Top 10% of customers' share of unplanned tickets", '47%', R.pct(top_obs)),
    ('Same share from their fleets plus chance (median of 1,000 draws)', '39%', R.pct(chance_share)),
    ('Rank correlation of customer STRs, first vs. second half', '0.46', R.num(rho, 2)),
    ("Same when shuffled (median of the script's 1,000 permutations, replayed with its seed)", '0.04',
     R.num(rho_null, 2)),
    ('Machines with 3 or more first-half tickets: second-half tickets vs. expected (times)', '2.6',
     R.num(machine_ratio, 1)),
    ('Consistent-outlier customers', '18', R.num(n_consistent_c)),
    ('Consistent-outlier machines', '36', R.num(n_consistent_m)),
    ("docs/05 chronic outliers on IPA's list", '106', R.num(n_on_list)),
    ('Of those, still flagged', '93', R.num(n_still_flagged)),
    ('Flagged machines that stand out under the negative binomial', 'no', 'no' if n_nb == 0 else R.num(n_nb)),
]);

,Finding,Report,Recomputed,Match
0,Fleet rate: unplanned tickets per 100 installed machine-months,6.2,6.2,yes
1,Three busiest models by unplanned tickets,"LVD-Remote-2.0, LVRR-WP-2.0, and MV-2.0","LVD-Remote-2.0, LVRR-WP-2.0, and MV-2.0",yes
2,Their share of installed machine-months,42%,42%,yes
3,Their share of unplanned tickets,51%,51%,yes
4,Their excess: raw vs. age-for-age ratio,partly because they are younger,partly because they are younger,yes
5,"scrubBank 480 vs. the fleet, age for age (times)",2.6,2.6,yes
6,Rate per 100: machines under 4 years,8,8,yes
7,Rate per 100: machines 10 years or more,2.5,2.5,yes
8,The 10+ rate vs. the drop with age (age-band factors with vs. without service status in the fit),overstates the drop with age,overstates the drop with age,yes
9,10+ machines with a ticket that are Not Serviceable,83,83,yes


All 25 values match the report.
